<a href="https://colab.research.google.com/github/Monicababu21/Chat-Conversation-Analyzer/blob/main/ChatConversationAnalyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install gradio pandas matplotlib

In [ ]:
import re
import os
import html
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from datetime import datetime

print("Chat Conversation Analyzer")
print("All libraries imported successfully!")

Chat Conversation Analyzer
All libraries imported successfully!


In [ ]:
SAMPLE_CONVERSATIONS = {
    "Team Discussion": """Monica: Hello everyone!
Arun: Hi Monica!
Priya: How is the project going?
Monica: The project is going well.
Arun: I completed the frontend today.
Priya: That is great!
Monica: Thank you for your help.
Arun: We should test the application tomorrow.
Priya: I agree with that plan.
Monica: Excellent work, everyone!""",

    "Project Discussion": """Monica: We need to finish the project today.
Arun: I am working on the backend.
Priya: The interface looks good.
Monica: Thank you for the update.
Arun: I found a problem in the database.
Priya: Do not worry, we can fix it.
Monica: Please share the error details.
Arun: I will send them shortly.
Priya: Great, let us solve this together.""",

    "Customer Feedback": """Customer: I really like this application.
Support: Thank you for your feedback.
Customer: The interface is easy to use.
Support: We are happy to hear that.
Customer: However, the application loads slowly.
Support: We will investigate the performance issue.
Customer: Thank you for your quick response.
Support: You are welcome."""
}


def load_sample(name):
    return SAMPLE_CONVERSATIONS.get(name, "")


print("Sample conversations are ready!")

Sample conversations are ready!


In [ ]:
def parse_conversation(text):
    messages = []

    # WhatsApp formats:
    # 22/10/26, 10:30 - Monica: Hello
    # [22/10/2026, 10:30:00] Monica: Hello
    whatsapp_pattern = re.compile(
        r"^\[?"
        r"(?P<date>\d{1,2}[/-]\d{1,2}[/-]\d{2,4})"
        r",\s*"
        r"(?P<time>\d{1,2}:\d{2}(?::\d{2})?\s?(?:AM|PM)?)"
        r"\]?\s*[-–]\s*"
        r"(?P<content>.*)$",
        re.IGNORECASE
    )

    # Simple format: Monica: Hello everyone!
    simple_pattern = re.compile(
        r"^(?P<speaker>[^:\n]{1,50}):\s*(?P<message>.+)$"
    )

    current = None

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        timestamp_match = whatsapp_pattern.match(line)

        if timestamp_match:
            content = timestamp_match.group("content")
            match = simple_pattern.match(content)

            if match:
                current = {
                    "timestamp": (
                        timestamp_match.group("date")
                        + " "
                        + timestamp_match.group("time")
                    ),
                    "speaker": match.group("speaker").strip(),
                    "message": match.group("message").strip()
                }
                messages.append(current)
            else:
                # Skip system notifications and lines without a speaker.
                current = None

            continue

        simple_match = simple_pattern.match(line)

        if simple_match:
            current = {
                "timestamp": None,
                "speaker": simple_match.group("speaker").strip(),
                "message": simple_match.group("message").strip()
            }
            messages.append(current)

        elif current is not None:
            # Keep multiline messages together.
            current["message"] += " " + line

    return pd.DataFrame(
        messages,
        columns=["timestamp", "speaker", "message"]
    )


print("Conversation parser is ready!")

Conversation parser is ready!


In [ ]:
POSITIVE_WORDS = {
    "good", "great", "excellent", "happy", "love", "like",
    "wonderful", "amazing", "helpful", "thanks", "thank",
    "beautiful", "best", "perfect", "success", "awesome",
    "enjoy", "glad", "fantastic", "appreciate", "well",
    "easy", "nice", "agree", "improve", "support"
}

NEGATIVE_WORDS = {
    "bad", "hate", "angry", "sad", "terrible", "awful",
    "poor", "worst", "problem", "issue", "slow", "wrong",
    "fail", "failed", "disappointed", "annoying", "difficult",
    "useless", "error", "broken", "frustrated", "unhappy",
    "dislike", "damage", "hurt", "negative"
}


def analyze_sentiment(message):
    words = re.findall(r"\b[a-zA-Z']+\b", message.lower())

    positive_count = sum(word in POSITIVE_WORDS for word in words)
    negative_count = sum(word in NEGATIVE_WORDS for word in words)

    if positive_count > negative_count:
        sentiment = "Positive"
    elif negative_count > positive_count:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    return sentiment


print("Sentiment analysis is ready!")

Sentiment analysis is ready!


In [ ]:
STOP_WORDS = {
    "the", "is", "a", "an", "and", "or", "but", "to", "of",
    "in", "on", "for", "with", "it", "this", "that", "we",
    "i", "you", "he", "she", "they", "are", "was", "were",
    "be", "been", "have", "has", "had", "do", "does", "did",
    "will", "would", "can", "could", "should", "please",
    "your", "our", "my", "me", "us", "them", "their", "at",
    "as", "by", "from", "if", "so", "not", "no", "yes",
    "very", "about", "we", "all", "everyone"
}


def get_word_frequency(messages, top_n=10):
    words = []

    for message in messages:
        tokens = re.findall(r"\b[a-zA-Z']+\b", message.lower())
        words.extend(
            word for word in tokens
            if len(word) > 2 and word not in STOP_WORDS
        )

    counts = Counter(words)

    return pd.DataFrame(
        counts.most_common(top_n),
        columns=["Word", "Frequency"]
    )


def get_conversation_statistics(df):
    if df.empty:
        return {}

    all_text = " ".join(df["message"].astype(str))
    words = re.findall(r"\b[\w'-]+\b", all_text)

    sentiment_counts = df["sentiment"].value_counts().to_dict()

    return {
        "Total Messages": len(df),
        "Total Participants": df["speaker"].nunique(),
        "Total Words": len(words),
        "Average Words per Message": round(
            len(words) / len(df), 2
        ),
        "Positive Messages": sentiment_counts.get("Positive", 0),
        "Negative Messages": sentiment_counts.get("Negative", 0),
        "Neutral Messages": sentiment_counts.get("Neutral", 0)
    }


print("Conversation statistics functions are ready!")

Conversation statistics functions are ready!


In [ ]:
def analyze_conversation(text, uploaded_file=None):
    if uploaded_file is not None:
        try:
            with open(uploaded_file, "r", encoding="utf-8-sig") as file:
                text = file.read()
        except Exception as error:
            return (
                "### File Error\nUnable to read the uploaded file.",
                pd.DataFrame(),
                pd.DataFrame(),
                pd.DataFrame(),
                None
            )

    if not text or not text.strip():
        return (
            "### No Conversation Provided",
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame(),
            None
        )

    df = parse_conversation(text)

    if df.empty:
        return (
            "### No Messages Found\nUse the format `Name: Message`.",
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame(),
            None
        )

    df["sentiment"] = df["message"].apply(analyze_sentiment)
    df["word_count"] = df["message"].apply(
        lambda message: len(message.split())
    )

    participant_df = (
        df.groupby("speaker")
        .agg(
            Messages=("message", "count"),
            Words=("word_count", "sum")
        )
        .reset_index()
        .rename(columns={"speaker": "Participant"})
        .sort_values("Messages", ascending=False)
    )

    frequency_df = get_word_frequency(df["message"].tolist())
    statistics = get_conversation_statistics(df)

    stats_df = pd.DataFrame(
        statistics.items(),
        columns=["Statistic", "Value"]
    )

    sentiment_counts = df["sentiment"].value_counts()

    summary = f"""
## 💬 Conversation Summary

- **Total messages:** {statistics["Total Messages"]}
- **Participants:** {statistics["Total Participants"]}
- **Total words:** {statistics["Total Words"]}
- **Average words per message:** {statistics["Average Words per Message"]}

### 😊 Sentiment Overview

- **Positive:** {statistics["Positive Messages"]}
- **Negative:** {statistics["Negative Messages"]}
- **Neutral:** {statistics["Neutral Messages"]}

*Sentiment is estimated using simple word matching. It may not reflect the actual meaning or intention of a message.*
"""

    report_path = "/content/chat_conversation_report.txt"

    with open(report_path, "w", encoding="utf-8") as file:
        file.write("CHAT CONVERSATION ANALYSIS REPORT\n")
        file.write(f"Generated: {datetime.now()}\n\n")
        file.write(summary + "\n\n")
        file.write("PARTICIPANT STATISTICS\n")
        file.write(participant_df.to_string(index=False))
        file.write("\n\nWORD FREQUENCY\n")
        file.write(frequency_df.to_string(index=False))
        file.write("\n\nMESSAGE-BY-MESSAGE ANALYSIS\n")
        file.write(
            df[["speaker", "message", "sentiment"]].to_string(index=False)
        )

    return summary, participant_df, stats_df, frequency_df, report_path


print("Main conversation analyzer is ready!")

Main conversation analyzer is ready!


In [ ]:
CSS = """
.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
}
body {
    background: #faf8ff;
}
h1, h2, h3 {
    color: #5b21b6;
}
button.primary {
    background: #6d28d9 !important;
    color: white !important;
}
"""

with gr.Blocks(title="Chat Conversation Analyzer") as app:

    gr.Markdown("""
    # 💬 Chat Conversation Analyzer
    ### Understand conversation patterns, participant activity, and sentiment.

    Paste a conversation or upload a TXT file.
    """)

    with gr.Tab("Conversation Analysis"):

        with gr.Row():

            with gr.Column():
                conversation_input = gr.Textbox(
                    label="Paste Conversation",
                    placeholder="Example:\nMonica: Hello!\nArun: Hi Monica!",
                    lines=12
                )

                sample_dropdown = gr.Dropdown(
                    choices=list(SAMPLE_CONVERSATIONS.keys()),
                    label="Choose Sample Conversation"
                )

                load_button = gr.Button("Load Sample")

                file_input = gr.File(
                    label="Or Upload a TXT File",
                    file_types=[".txt"],
                    type="filepath"
                )

                with gr.Row():
                    analyze_button = gr.Button(
                        "Analyze Conversation",
                        variant="primary"
                    )
                    clear_button = gr.ClearButton(
                        [conversation_input, file_input]
                    )

            with gr.Column():
                summary_output = gr.Markdown(
                    label="Analysis Summary"
                )

                report_output = gr.File(
                    label="Download Report"
                )

        gr.Markdown("## 👥 Participant Statistics")

        participant_output = gr.Dataframe(
            label="Participant Activity",
            interactive=False
        )

        gr.Markdown("## 📊 Conversation Statistics")

        stats_output = gr.Dataframe(
            label="Statistics",
            interactive=False
        )

        gr.Markdown("## 🔑 Frequently Used Words")

        frequency_output = gr.Dataframe(
            label="Word Frequency",
            interactive=False
        )

        load_button.click(
            fn=load_sample,
            inputs=sample_dropdown,
            outputs=conversation_input
        )

        analyze_button.click(
            fn=analyze_conversation,
            inputs=[conversation_input, file_input],
            outputs=[
                summary_output,
                participant_output,
                stats_output,
                frequency_output,
                report_output
            ]
        )

    with gr.Tab("How It Works"):
        gr.Markdown("""
        ## Analysis Process

        1. **Message Parsing:** Identifies speakers and messages.
        2. **Participant Analysis:** Counts each participant's messages and words.
        3. **Sentiment Analysis:** Estimates sentiment using positive and negative word lists.
        4. **Word Frequency:** Identifies commonly used meaningful words.
        5. **Statistics:** Calculates conversation metrics.
        6. **Report Generation:** Produces a downloadable text report.

        ### Limitations
        - Messages should use a recognizable speaker format.
        - Sentiment analysis is rule-based and can misunderstand context.
        - Timestamp extraction is supported for common WhatsApp formats.
        - The prototype does not interpret emojis or reliably detect sarcasm.
        """)

    with gr.Tab("About"):
        gr.Markdown("""
        ## About the Project

        The Chat Conversation Analyzer is a Python-based Natural Language
        Processing application for analyzing text conversations.

        **Technologies Used**
        - Python
        - Pandas
        - Regular Expressions
        - Matplotlib
        - Gradio

        **Applications**
        - Team communication analysis
        - Customer feedback review
        - Chat activity statistics
        - Basic conversation sentiment analysis
        """)

app.launch(share=True, css=CSS)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0e98fe4e919ac45e7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
